In [1]:
%pip install matplotlib


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import math
import sys

# Complete inference

## Tokenizer

In [3]:
import json
import regex

class Tokenizer:

    def __init__(self, tokenizer_path: str):
        with open(tokenizer_path) as f:
            tokenizer_data = json.load(f)
        split = next(filter(lambda t: t["type"] == "Split", tokenizer_data["pre_tokenizer"]["pretokenizers"]))
        self.split_regex = regex.compile(split["pattern"]["Regex"])

        # space is encoded as Ġ, for simplicity, just use space here.
        self.vocab = {k.replace("Ġ", " ").encode("utf-8"): v for k, v in tokenizer_data["model"]["vocab"].items()}
        added_tokens = {t["content"]: t["id"] for t in tokenizer_data["added_tokens"]}

        self.begin_of_text = added_tokens["<|begin_of_text|>"]
        self.end_of_text = added_tokens["<|end_of_text|>"]

        self.vocab.update(added_tokens)

        # inverse vocabulary for detokenization
        self.vocab_inv = { v: k for k, v in self.vocab.items() }

    def tokenize(self, text: str) -> list[int]:
        str_tokens = self.split_regex.findall(text)

        # Add specific markers for beginning and end of text
        # str_tokens = ["<|begin_of_text|>"] + str_tokens + ["<|end_of_text|>"]

        tokens = []

        for str_token in str_tokens:
            parts = [bytes([b]) for b in str_token.encode("utf-8")]

            while True:
                # Iterate over all pairs and find the pair we want to merge the most
                min_idx = None
                min_rank = None
                for i, pair in enumerate(zip(parts[:-1], parts[1:])):
                    rank = self.vocab.get(pair[0] + pair[1])
                    if rank is not None and (min_rank is None or rank < min_rank):
                        min_idx = i
                        min_rank = rank

                # If there were no pairs we could merge, we're done!
                if min_rank is None:
                    break
                assert min_idx is not None

                # Otherwise, merge that pair and leave the rest unchanged. Then repeat.
                parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2 :]

            tokens.extend(self.vocab[part] for part in parts)

        return [self.begin_of_text] + tokens

    def detokenize(self, tokens: list[int]) -> str:
        decoded = b""
        for t in tokens:
            decoded += self.vocab_inv[t]

        return str(decoded, "utf-8")

## Model weights

Ensure that the content of `1-fetch-files.ipynb` has been executed to fetch and convert the model weights from Huggingface

In [4]:
from pathlib import Path
import mmap
import numpy as np
from numpy.typing import NDArray

def load_raw_model(path: str):
    model_dir = Path(path)
    model = {}
    with open(model_dir / "metadata.json") as file:
        metadata = json.load(file)
    for tensor_name, tensor_metadata in metadata.items():
        if tensor_name == "__metadata__":
            continue
        file = open(model_dir / f"{tensor_name}.raw", mode="rb")
        mmaped = mmap.mmap(file.fileno(), 0, prot=mmap.PROT_READ)
        model[tensor_name] = np.frombuffer(mmaped, dtype=np.float32).reshape(tensor_metadata["shape"])
    return model

## Network blocks

### Helper functions

In [5]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [6]:
def silu(x):
    """
    Sigmoid Linear Unit
    The SiLU function is also known as the swish function.
    """
    return x * (1.0 / (1.0 + np.exp(-x)))

### Attention block

In [ ]:
from typing import Any


class AttentionBlock:
    def __init__(
        self,
        layer_index: int,
        config: dict[str, Any],
        weights: dict[str, NDArray],
        freqs_cos: NDArray,
        freqs_sin: NDArray,
    ):
        self.freqs_cos = freqs_cos
        self.freqs_sin = freqs_sin

        self.num_key_value_heads = config["num_key_value_heads"]
        self.num_attention_heads = config["num_attention_heads"]
        self.head_dim = config["head_dim"]
        self.max_seq_len = config["max_seq_len"]

        self.q_weight = weights[f"model.layers.{layer_index}.self_attn.q_proj.weight"].T
        self.k_weight = weights[f"model.layers.{layer_index}.self_attn.k_proj.weight"].T
        self.v_weight = weights[f"model.layers.{layer_index}.self_attn.v_proj.weight"].T
        self.o_weight = weights[f"model.layers.{layer_index}.self_attn.o_proj.weight"].T

        self.k_cache = np.zeros((self.max_seq_len, self.num_key_value_heads, self.head_dim))
        self.v_cache = np.zeros((self.max_seq_len, self.num_key_value_heads, self.head_dim))


    def apply_rotary_positional_encoding(
        self,
        xq: NDArray,
        xk: NDArray,
        pos: int,
    ) -> tuple[NDArray, NDArray]:
        freqs_cos = self.freqs_cos[pos]
        freqs_sin = self.freqs_sin[pos]

        # NOTE: here the implementation of Llama differs from what can be found in other transformer
        # implementations (ex Karpathy's). Embedding value pairs are considered as complex numbers
        # Here, the first half of the embedding contains the real parts, the second half, the imaginary parts

        xq_r, xq_i = np.split(xq, 2, axis=-1)
        xk_r, xk_i = np.split(xk, 2, axis=-1)

        freqs_cos = np.expand_dims(freqs_cos, axis=0)
        freqs_sin = np.expand_dims(freqs_sin, axis=0)

        # Apply rotation using complex numbers.
        xq_out_r = xq_r * freqs_cos - xq_i * freqs_sin
        xq_out_i = xq_r * freqs_sin + xq_i * freqs_cos
        xk_out_r = xk_r * freqs_cos - xk_i * freqs_sin
        xk_out_i = xk_r * freqs_sin + xk_i * freqs_cos

        xq_out = np.concatenate([xq_out_r, xq_out_i], axis=-1)
        xk_out = np.concatenate([xk_out_r, xk_out_i], axis=-1)

        return xq_out, xk_out

    def __call__(self, x: NDArray, pos: int) -> NDArray:
        q = x @ self.q_weight
        k = x @ self.k_weight
        v = x @ self.v_weight

        # Compute query, key and value for the given token
        q = q.reshape(self.num_attention_heads, self.head_dim)
        k = k.reshape(self.num_key_value_heads, self.head_dim)
        v = v.reshape(self.num_key_value_heads, self.head_dim)

        # Apply positional encoding
        q, k = self.apply_rotary_positional_encoding(q, k, pos)

        # Populate KV cache
        self.k_cache[pos] = k
        self.v_cache[pos] = v

        # Extract all key and values from the beginning up to the current one from the cache.
        keys = self.k_cache[: pos + 1]
        values = self.v_cache[: pos + 1]

        repeats = self.num_attention_heads // self.num_key_value_heads
        keys = np.repeat(keys, repeats, axis=1)
        values = np.repeat(values, repeats, axis=1)

        head_outputs = np.zeros((self.num_attention_heads, self.head_dim))

        # Compute attention independently for each head.
        for head_index in range(self.num_attention_heads):
            q_head = q[head_index]
            k_head = keys[:, head_index, :]
            v_head = values[:, head_index, :]

            # Compute similarity scores between the current query and all cached keys
            # up to the current position.
            scores = (k_head @ q_head) / math.sqrt(self.head_dim)
            # Normalize scores into attention weights
            attn_weights = softmax(scores)
            # Compute a weighted sum of values with attention weights
            head_outputs[head_index] = attn_weights @ v_head

        combined = head_outputs.reshape(-1)
        # Project the concatenated head outputs with the output weight matrix.
        output = combined @ self.o_weight
        return output

### Other blocks

In [8]:
class FeedForward:
    def __init__(self, layer_index: int, weights: dict[str, NDArray]):
        self.up_weight = weights[f"model.layers.{layer_index}.mlp.up_proj.weight"].T
        self.down_weight = weights[f"model.layers.{layer_index}.mlp.down_proj.weight"].T
        self.gate_weight = weights[f"model.layers.{layer_index}.mlp.gate_proj.weight"].T

    def __call__(self, x: NDArray) -> NDArray:
        gate = silu(x @ self.gate_weight)
        up = x @ self.up_weight
        fused = gate * up
        out = fused @ self.down_weight
        return out

In [9]:
class RMSNorm:
    def __init__(self, weights_array: NDArray, eps: float):
        self.weights = weights_array
        self.eps = eps

    def __call__(self, x: NDArray) -> NDArray:
        rms = np.sqrt(np.mean(x ** 2) + self.eps)
        return (x / rms) * self.weights

### Transformer block

In [10]:
class TransformerBlock:
    def __init__(
        self,
        layer_index: int,
        config: dict[str, Any],
        weights: dict[str, NDArray],
        freqs_cos: NDArray,
        freqs_sin: NDArray,
    ):
        self.attention = AttentionBlock(layer_index, config, weights, freqs_cos, freqs_sin)
        self.feed_forward = FeedForward(layer_index, weights)

        self.input_layernorm = RMSNorm(
            weights[f"model.layers.{layer_index}.input_layernorm.weight"],
            eps=config["rms_norm_eps"],
        )
        self.post_attention_layernorm = RMSNorm(
            weights[f"model.layers.{layer_index}.post_attention_layernorm.weight"],
            eps=config["rms_norm_eps"],
        )

    def __call__(self, x: NDArray, start_pos: int) -> NDArray:
        # Normalize the input with the pre‑attention layer norm
        norm_x = self.input_layernorm(x)

        # Apply self‑attention
        attn_out = self.attention(norm_x, start_pos)
        # Add the attention output to the residual connection
        x = x + attn_out

        # Normalize the post‑attention
        norm_x = self.post_attention_layernorm(x)

        ff_out = self.feed_forward(norm_x)
        # Add the feed‑forward output to the residual connection
        x = x + ff_out

        return x

# Llama network

In [ ]:

class Llama:
    def __init__(self, model_path: str, config_path: str, max_seq_len: int):
        weights = load_raw_model(model_path)

        with open(config_path) as file:
            config = json.load(file)

        self.bos_token_id = config["bos_token_id"]

        config["max_seq_len"] = max_seq_len
        config["head_dim"] = config["hidden_size"] // config["num_attention_heads"]

        self.token_embeddings = weights["model.embed_tokens.weight"]

        freqs_cos, freqs_sin = self.compute_cos_sin_cache(
            head_dim=config["head_dim"],
            max_seq_len=config["max_seq_len"],
            base=config["rope_theta"],
        )

        self.layers = [
            TransformerBlock(layer_index, config, weights, freqs_cos, freqs_sin)
            for layer_index in range(config["num_hidden_layers"])
        ]

        self.norm = RMSNorm(weights["model.norm.weight"], eps=config["rms_norm_eps"])
        self.lm_head_weight = weights["model.embed_tokens.weight"].T

    def compute_cos_sin_cache(
        self,
        head_dim: int,
        max_seq_len: int,
        base: float,
    ) -> tuple[NDArray, NDArray]:
        inv_freq = 1.0 / (base ** (np.arange(0, head_dim, 2) / head_dim))
        positions = np.arange(max_seq_len)
        angles = np.outer(positions, inv_freq)
        return np.cos(angles), np.sin(angles)

    def __call__(self, input_id: int, start_pos: int) -> NDArray:
        h = self.token_embeddings[input_id]

        # Run the token representation through all transformer layers.
        for layer in self.layers:
            h = layer(h, start_pos)

        h = self.norm(h)
        logits = h @ self.lm_head_weight
        return logits

    def generate(self, input_ids: NDArray, max_new_tokens: int):
        # Prefill and Decode
        #
        # Inference happens in two phases.
        # During the prefill phase, the model processes the full prompt token by token
        # and stores keys and values in the KV cache for each layer.
        # During the decode phase, the model generates one new token at a time. At each step,
        # it reuses the cached keys and values from previous tokens instead of recomputing
        # the whole prompt.
        
        prompt_length = len(input_ids)

        if prompt_length > 0:
            # Prefill phase. This populates each transformer layer's KV cache
            # with the prompt tokens.
            for pos in range(prompt_length):
                logits = self(int(input_ids[pos]), pos)
            next_id = int(np.argmax(logits))
        else:
            next_id = self.bos_token_id

        # Generation phase
        for pos in range(prompt_length, prompt_length + max_new_tokens):
            logits = self(next_id, pos)
            next_id = int(np.argmax(logits))
            yield next_id

In [12]:
tokenizer = Tokenizer("models/Llama-3.2-1B/tokenizer.json")
llama = Llama("models/Llama-3.2-1B/tensors-fp32", "models/Llama-3.2-1B/config.json", 200)

In [13]:
prompt = "Holy cow, it's working"

In [14]:
input_ids = np.array([tokenizer.tokenize(prompt)])[0]

In [15]:
display(input_ids.shape)
display(input_ids)

(7,)

array([128000,  72291,  19923,     11,    433,    596,   3318])

In [ ]:
print(prompt, end="")

for id in llama.generate(input_ids, max_new_tokens=150):
    if id == tokenizer.end_of_text:
        break
    print(tokenizer.detokenize([id]), end="")
    sys.stdout.flush()


Holy cow, it's working I'm so excited to be able to